# Multicoil + time NIK debugInteractive prototype mirroring `train_multicoil_cart.py`: WIRE_KXY_COIL_T model, focal loss, shared envelope, SENSE-combined per-frame metrics.

In [ ]:
import numpy as npimport torchimport torch.nn.functional as Fimport matplotlib.pyplot as pltfrom nik_io import load_event, synthesize_cartesian_from_radialfrom nik_train import prepare_tensorsfrom nik_recon import (    ifft1d_kz_to_z,    ifft1d_kz_to_z_cartesian,    make_multicoil_time_radial_dataset,    make_multicoil_time_cartesian_dataset,    coil_combine_rss,)from kspace_normalization import compute_dcf_radial, compute_radius, KSpaceNormalizerfrom nik_focal_loss import composable_kspace_loss, _residual_magsqfrom nik_model import WIRE_KXY_COIL_T_REIMfrom nik_metrics import compute_image_metrics, compute_perceptual_metricsdevice = 'cuda' if torch.cuda.is_available() else 'cpu'print('device:', device)

## Manual params

In [ ]:
radial_file = '/scratch/rnga/vvpshenov/XCAT-ERIC/results/simulation_results_20260326T114914_rad.mat'z_slice_raw = -1subsample_frac = 0.7seed = 0# modelhidden, depth, w0, s0 = 96, 8, 62.0, 10.0coil_embed_dim = 8dropout = 0.0# trainingsteps        = 5000           # short for interactive; bump for real runsbatch_size   = 8192lr           = 1e-4weight_decay = 3e-3grad_clip    = 1.0eval_every   = 100warmup_steps = 1000           # for best-state restoreconsole_every = 500# loss flags (cell D + alpha=1.5 from prior sweeps)use_envelope     = Trueuse_dcf          = Falsedcf_power        = 0.0use_focal        = Truefocal_alpha      = 1.5focal_normalize  = Truefocal_log_matrix = Falsefocal_warmup_steps = 500# plateau scheduler driven by heldoutsched_patience = 10sched_factor   = 0.5sched_min_lr   = 1e-6torch.manual_seed(seed); np.random.seed(seed)

## Load radial data

In [ ]:
event = load_event(radial_file, load_images=True, load_coil_maps=True)k_np    = np.transpose(event['k'],    (0, 2, 1, 3))traj_np = np.transpose(event['traj'], (0, 2, 1, 3))T, S, C, RO = k_np.shapeprint(f'k_np {k_np.shape} {k_np.dtype}')print(f'T={T}  S={S}  C={C}  RO={RO}')k_t, traj_t, scales, dims, k_scale = prepare_tensors(k_np, traj_np, data_device=device)k_img_space, n_z_slices, n_ro_per_slice, _ = ifft1d_kz_to_z(k_t, traj_t, t_frame=0)z_slice_idx = n_z_slices // 2 if z_slice_raw == -1 else int(z_slice_raw)print(f'n_z_slices={n_z_slices}  n_ro_per_slice={n_ro_per_slice}  z_slice_idx={z_slice_idx}')

## Multicoil + time radial dataset (training)

In [ ]:
(    x_all, t_all, coil_all, y_all_raw,    spoke_id_all, ro_id_all, frame_id_all, meta_rad,) = make_multicoil_time_radial_dataset(    k_img_space, traj_t, scales, dims,    z_slice_idx=z_slice_idx, n_slices=n_z_slices, compute_device=device,)print(f'x_all     {x_all.shape}      dtype={x_all.dtype}')print(f't_all     {t_all.shape}      range [{t_all.min().item():.3f}, {t_all.max().item():.3f}]')print(f'coil_all  {coil_all.shape}   unique={coil_all.unique().tolist()}')print(f'y_all_raw {y_all_raw.shape}  dtype={y_all_raw.dtype}')print(f'spokes={int(spoke_id_all.max())+1}  frames={int(frame_id_all.max())+1}  N={meta_rad["N"]}')

## Synthesize Cartesian (all T, all C) + dataset

In [ ]:
cart_event = synthesize_cartesian_from_radial(radial_file, T_target=T, event=event)k_cart_np = cart_event['k_cart']coil_maps_cart = cart_event['coil_maps']k_cart_t = torch.from_numpy(k_cart_np.astype(np.complex64)).to(device)k_cart_z = ifft1d_kz_to_z_cartesian(k_cart_t)z_slice_cart = k_cart_z.shape[2] // 2 if z_slice_raw == -1 else int(z_slice_raw)print(f'k_cart   {k_cart_t.shape}')print(f'k_cart_z {k_cart_z.shape}')print(f'z_slice_cart={z_slice_cart}')(    x_cart, t_cart, coil_cart, y_cart_raw, frame_id_cart, meta_cart,) = make_multicoil_time_cartesian_dataset(    k_cart_z, z_slice_idx=z_slice_cart, scales_radial=scales, compute_device=device,)nky, nkx = meta_cart['nky'], meta_cart['nkx']print(f'x_cart     {x_cart.shape}')print(f't_cart     range [{t_cart.min().item():.3f}, {t_cart.max().item():.3f}]')print(f'meta_cart  {meta_cart}')

## Sanity: round-trip ifft(k_cart) == coils * gt * sp

In [ ]:
sp = cart_event['SP']gt_pad = cart_event['gt_img']         # (T, kz, RL_pad, AP_pad)coil_maps = cart_event['coil_maps']   # (C, kz, RL_pad, AP_pad)t_test = 0rec = np.fft.ifftn(np.fft.ifftshift(k_cart_np[t_test], axes=(1,2,3)), axes=(1,2,3))rec_coil_axes = rec.transpose(0, 1, 3, 2)   # (C, kz, RL_pad, AP_pad)gt_truth = coil_maps * gt_pad[t_test:t_test+1] * sp.reshape(1, -1, 1, 1)err = np.abs(rec_coil_axes - gt_truth)print(f'max abs err : {err.max():.3e}')print(f'rel err     : {err.max() / (np.abs(gt_truth).max() + 1e-12):.3e}')

## Train / heldout spoke split (shared across t,c)

In [ ]:
n_unique_spokes = int(spoke_id_all.max().item()) + 1n_train_spokes  = max(1, int(n_unique_spokes * subsample_frac))g = torch.Generator(device=spoke_id_all.device).manual_seed(seed)perm = torch.randperm(n_unique_spokes, generator=g, device=spoke_id_all.device)train_spokes  = perm[:n_train_spokes]train_mask = torch.isin(spoke_id_all, train_spokes)train_idx  = torch.where(train_mask)[0]heldout_idx = torch.where(~train_mask)[0]print(f'spokes {n_train_spokes}/{n_unique_spokes} ({subsample_frac:.0%})')print(f'train pts {train_idx.numel()}   heldout pts {heldout_idx.numel()}')

## DCF + one shared envelope (fit on train pool)

In [ ]:
kcoords        = x_allkcoords_train  = kcoords[train_idx]y_train_raw    = y_all_raw[train_idx]dcf = compute_dcf_radial(kcoords) if use_dcf else torch.ones(kcoords.shape[0], device=kcoords.device)dcf_train_norm = dcf[train_idx]normalizer = KSpaceNormalizer()if use_envelope:    normalizer.fit(        kcoords_train, y_train_raw, dcf=dcf_train_norm,        envelope_bins=128, envelope_statistic='weighted_rms',        envelope_smooth_method='moving_average', envelope_smooth_width=5,        envelope_floor_fraction=1e-3, global_scale_method='weighted_rms',    )else:    from kspace_normalization import compute_global_scale, _to_complex, RadialEnvelope    y_c = _to_complex(y_train_raw)    normalizer.global_scale = compute_global_scale(y_c, dcf=dcf_train_norm)    r_max = float(compute_radius(kcoords_train).max().item())    normalizer.envelope = RadialEnvelope(        bin_centers=torch.linspace(0, r_max, 128),        raw_shell_values=torch.ones(128), smoothed_shell_values=torch.ones(128),        floor_value=1.0, r_max=r_max, statistic='flat', smooth_method='none')    normalizer._fitted = Truey_all  = normalizer.normalize(kcoords, y_all_raw)y_cart = normalizer.normalize(x_cart,  y_cart_raw)print(f'global_scale = {normalizer.global_scale:.4f}')print(f'use_envelope={use_envelope}  use_dcf={use_dcf}  dcf_power={dcf_power}')print(f'use_focal={use_focal}  alpha={focal_alpha}  log_matrix={focal_log_matrix}  warmup={focal_warmup_steps}')

## Model: WIRE_KXY_COIL_T_REIM

In [ ]:
model = WIRE_KXY_COIL_T_REIM(    n_coils=C, coil_embed_dim=coil_embed_dim,    hidden=hidden, depth=depth, w0=w0, s0=s0, dropout=dropout,).to(device)n_params = sum(p.numel() for p in model.parameters())print(f'params: {n_params}')print(f'in_dim = 2 (kxy) + 1 (t) + {coil_embed_dim} (coil) = {2+1+coil_embed_dim}')# forward smokewith torch.no_grad():    yp = model(x_all[:8], t_all[:8], coil_all[:8])print('forward smoke:', yp.shape, yp.dtype)

## Training (focal loss, plateau scheduler on heldout)

In [ ]:
# device tensorsx_all_dev    = x_all.to(device)t_all_dev    = t_all.to(device)coil_all_dev = coil_all.to(device)y_all_dev    = y_all.to(device)dcf_dev      = dcf.to(device)x_train    = x_all_dev[train_idx]t_train    = t_all_dev[train_idx]coil_train = coil_all_dev[train_idx]y_train    = y_all_dev[train_idx]dcf_train  = dcf_dev[train_idx]N_train    = x_train.shape[0]x_held    = x_all_dev[heldout_idx]t_held    = t_all_dev[heldout_idx]coil_held = coil_all_dev[heldout_idx]y_held    = y_all_dev[heldout_idx]# cart disk maskradial_rmax = float(compute_radius(kcoords_train).max().item())cart_r      = compute_radius(x_cart.to(device))cart_in_disk_mask = cart_r <= (radial_rmax + 1e-6)x_cart_dev    = x_cart.to(device)t_cart_dev    = t_cart.to(device)coil_cart_dev = coil_cart.to(device)y_cart_dev    = y_cart.to(device)opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(    opt, mode='min', factor=sched_factor, patience=sched_patience, min_lr=sched_min_lr,)# historytrain_losses, steps_v, val_held, val_cart_disk = [], [], [], []best_held = float('inf'); best_step = -1; best_state = Nonefocal_diags = []   # diagnostic dicts at each eval stepmodel.train()for step in range(1, steps + 1):    idx = torch.randint(0, N_train, (batch_size,), device=device)    xb  = x_train[idx]    tb  = t_train[idx]    cb  = coil_train[idx]    yb  = y_train[idx]    wb  = dcf_train[idx]    opt.zero_grad(set_to_none=True)    yp = model(xb, tb, cb)    fp = min(1.0, step / float(focal_warmup_steps)) if focal_warmup_steps > 0 else 1.0    want_diag = (step % eval_every == 0 or step == steps or step == 1)    if want_diag:        loss, fdiag = composable_kspace_loss(            yp, yb,            dcf=wb, use_dcf=use_dcf, dcf_power=dcf_power,            use_focal=use_focal, focal_alpha=focal_alpha,            focal_normalize=focal_normalize, focal_log_matrix=focal_log_matrix,            focal_warmup_progress=fp,            return_diagnostics=True,        )    else:        loss = composable_kspace_loss(            yp, yb,            dcf=wb, use_dcf=use_dcf, dcf_power=dcf_power,            use_focal=use_focal, focal_alpha=focal_alpha,            focal_normalize=focal_normalize, focal_log_matrix=focal_log_matrix,            focal_warmup_progress=fp,            return_diagnostics=False,        )        fdiag = None    loss.backward()    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)    opt.step()    train_losses.append(float(loss.item()))    if step % eval_every == 0 or step == steps:        model.eval()        with torch.no_grad():            held_pred = model(x_held, t_held, coil_held)            cur_held  = float(F.mse_loss(held_pred, y_held).item())            cart_pred = model(x_cart_dev, t_cart_dev, coil_cart_dev)            cur_cart  = float(F.mse_loss(cart_pred[cart_in_disk_mask], y_cart_dev[cart_in_disk_mask]).item())        model.train()        steps_v.append(step); val_held.append(cur_held); val_cart_disk.append(cur_cart)        if fdiag is not None:            fdiag['step'] = step            focal_diags.append(fdiag)        # heldout-driven plateau + best-state        scheduler.step(cur_held)        if step >= warmup_steps and cur_held < best_held:            best_held, best_step = cur_held, step            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}    if step % console_every == 0 or step == 1:        msg = f'step {step:6d}  train {train_losses[-1]:.3e}'        if val_held:      msg += f'  held {val_held[-1]:.3e}'        if val_cart_disk: msg += f'  cart_disk {val_cart_disk[-1]:.3e}'        msg += f'  lr {opt.param_groups[0]["lr"]:.1e}'        print(msg)if best_state is not None:    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})    print(f'restored best @ step {best_step}, heldout = {best_held:.4e}')else:    print('no best checkpoint (warmup_steps not crossed)')

### Training curves

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))ax[0].plot(train_losses, alpha=0.6, lw=0.7)ax[0].set_yscale('log'); ax[0].set_xlabel('step'); ax[0].set_title('train loss (focal)')ax[1].plot(steps_v, val_held, 's-', ms=3); ax[1].set_yscale('log')ax[1].set_xlabel('step'); ax[1].set_title('val heldout (plain MSE)')ax[2].plot(steps_v, val_cart_disk, 'o-', ms=3); ax[2].set_yscale('log')ax[2].set_xlabel('step'); ax[2].set_title('val cart in-disk (plain MSE)')if best_step > 0:    for a in ax[1:]: a.axvline(best_step, color='r', ls='--', lw=0.7, alpha=0.7)plt.tight_layout(); plt.show()if focal_diags:    fig, ax = plt.subplots(1, 2, figsize=(10, 3))    fd_steps = [d['step'] for d in focal_diags]    ax[0].plot(fd_steps, [d['w_focal_max']  for d in focal_diags], label='max')    ax[0].plot(fd_steps, [d['w_focal_mean'] for d in focal_diags], label='mean')    ax[0].plot(fd_steps, [d['w_focal_p99']  for d in focal_diags], label='p99')    ax[0].set_yscale('log'); ax[0].legend(); ax[0].set_title('w_focal stats'); ax[0].set_xlabel('step')    ax[1].plot(fd_steps, [d['top1pct_loss_frac'] for d in focal_diags])    ax[1].set_title('top-1% loss contribution'); ax[1].set_xlabel('step')    plt.tight_layout(); plt.show()

## Per-frame SENSE-combined recon + image metrics

In [ ]:
model.eval()with torch.no_grad():    cart_pred_norm   = model(x_cart_dev, t_cart_dev, coil_cart_dev)    cart_pred_denorm = normalizer.denormalize(x_cart_dev, cart_pred_norm)    y_meas_denorm    = normalizer.denormalize(x_cart_dev, y_cart_dev)pred_r = cart_pred_denorm[:, 0].view(T, C, nky, nkx).cpu().numpy()pred_i = cart_pred_denorm[:, 1].view(T, C, nky, nkx).cpu().numpy()meas_r = y_meas_denorm[:, 0].view(T, C, nky, nkx).cpu().numpy()meas_i = y_meas_denorm[:, 1].view(T, C, nky, nkx).cpu().numpy()k_pred_TC = pred_r + 1j * pred_ik_meas_TC = meas_r + 1j * meas_isens = coil_maps_cart[:, z_slice_cart, :, :].astype(np.complex64)   # (C, H, W)print('sens shape:', sens.shape)norm_im = lambda a: a / (a.max() + 1e-12)per_frame = []pred_imgs, meas_imgs = [], []for t_idx in range(T):    pred_coil_imgs = np.stack([np.fft.fftshift(np.fft.ifft2(k_pred_TC[t_idx, c])).T for c in range(C)], axis=0)    meas_coil_imgs = np.stack([np.fft.ifft2(k_meas_TC[t_idx, c]).T                  for c in range(C)], axis=0)    if sens.shape == pred_coil_imgs.shape:        denom = np.sum(np.abs(sens) ** 2, axis=0) + 1e-10        img_pred = np.abs(np.sum(np.conj(sens) * pred_coil_imgs, axis=0) / denom)        img_meas = np.abs(np.sum(np.conj(sens) * meas_coil_imgs, axis=0) / denom)    else:        img_pred = coil_combine_rss(pred_coil_imgs)        img_meas = coil_combine_rss(meas_coil_imgs)    img_pred_n = norm_im(img_pred); img_meas_n = norm_im(img_meas)    m = compute_image_metrics(img_pred_n, img_meas_n)    p = compute_perceptual_metrics(img_pred_n, img_meas_n)    per_frame.append({'t': t_idx, 'psnr': m['psnr_db'], 'ssim': m['ssim'],                      'dists': p['DISTS'], 'haarpsi': p['HaarPSI']})    pred_imgs.append(img_pred_n); meas_imgs.append(img_meas_n)print(f"{'t':>3} {'PSNR':>7} {'SSIM':>7} {'DISTS':>7} {'HaarPSI':>8}")for f in per_frame:    print(f"{f['t']:>3} {f['psnr']:>7.3f} {f['ssim']:>7.4f} {f['dists']:>7.4f} {f['haarpsi']:>8.4f}")m_psnr    = float(np.mean([f['psnr']    for f in per_frame]))m_ssim    = float(np.mean([f['ssim']    for f in per_frame]))m_dists   = float(np.mean([f['dists']   for f in per_frame]))m_haarpsi = float(np.mean([f['haarpsi'] for f in per_frame]))print(f'\nMEAN  PSNR={m_psnr:.3f}  SSIM={m_ssim:.4f}  DISTS={m_dists:.4f}  HaarPSI={m_haarpsi:.4f}')

### Visualize each frame: pred / meas / |diff|

In [ ]:
fig, ax = plt.subplots(3, T, figsize=(3*T, 8))ax = np.atleast_2d(ax)for t_idx in range(T):    ax[0, t_idx].imshow(pred_imgs[t_idx], cmap='gray');                              ax[0, t_idx].set_title(f't={t_idx}  pred', fontsize=9); ax[0, t_idx].axis('off')    ax[1, t_idx].imshow(meas_imgs[t_idx], cmap='gray');                              ax[1, t_idx].set_title(f't={t_idx}  ref',  fontsize=9); ax[1, t_idx].axis('off')    diff = np.abs(pred_imgs[t_idx] - meas_imgs[t_idx])    ax[2, t_idx].imshow(diff, cmap='hot', vmax=0.3);                                  ax[2, t_idx].set_title(f't={t_idx}  |pred-ref|', fontsize=9); ax[2, t_idx].axis('off')plt.tight_layout(); plt.show()# per-frame metric trendsfig, ax = plt.subplots(1, 3, figsize=(12, 3))xs = list(range(T))ax[0].plot(xs, [f['psnr']    for f in per_frame], 'o-'); ax[0].set_title('PSNR vs frame'); ax[0].set_xlabel('t')ax[1].plot(xs, [f['dists']   for f in per_frame], 's-'); ax[1].set_title('DISTS vs frame'); ax[1].set_xlabel('t')ax[2].plot(xs, [f['haarpsi'] for f in per_frame], '^-'); ax[2].set_title('HaarPSI vs frame'); ax[2].set_xlabel('t')plt.tight_layout(); plt.show()

### Sanity: per-coil pred vs meas at one frame

In [ ]:
t_pick = T // 2fig, ax = plt.subplots(2, C, figsize=(2.2*C, 4.5))for c in range(C):    img_pred_c = np.abs(np.fft.fftshift(np.fft.ifft2(k_pred_TC[t_pick, c])).T)    img_meas_c = np.abs(np.fft.ifft2(k_meas_TC[t_pick, c]).T)    ax[0, c].imshow(img_pred_c, cmap='gray'); ax[0, c].set_title(f'pred c{c}', fontsize=8); ax[0, c].axis('off')    ax[1, c].imshow(img_meas_c, cmap='gray'); ax[1, c].set_title(f'meas c{c}', fontsize=8); ax[1, c].axis('off')plt.suptitle(f'per-coil at t={t_pick}', y=1.02, fontsize=10)plt.tight_layout(); plt.show()